# KinoCompass – итоговый тест, часть на GPU

**Тест – один раз.** Модели и настройки выбраны на валидации и зафиксированы в репозитории (шаг 6в); здесь они только переобучаются на обучении + валидации (всё до 14.10.2022) и проверяются на последнем годе (14.10.2022 – 13.10.2023). Повторно не запускать и ничего не подбирать по результату.

Состав (`python evaluate.py final_gpu`): популярные за последний год (для парных сравнений), EASE λ = 5000, Mult-VAE («вкус» 400, β 0,5) и смесь «популярность × Mult-VAE» с весом 0,2. Остальные модели – в Docker (`python evaluate.py final`), те же люди и сценарии: модели сравниваются парным тестом между прогонами.

**Как запустить:** «Среда выполнения» → «Сменить среду выполнения» → **T4 GPU**, затем «Среда выполнения» → «Выполнить все». В конце скачается архив `final_gpu_test.zip` – его нужно положить в `model/reports/`.

Время на T4 (оценка): загрузка набора и сценарии – около 3 минут, EASE – 2–5 минут, Mult-VAE – около 8 минут, проверка – 5–10 минут; всего 20–30 минут. Вкладку не закрывать до конца.

In [ ]:
# итоговый тест – один раз (модели выбраны на валидации, шаг 6в)
PERIOD = "test"

In [ ]:
import torch

assert torch.cuda.is_available(), "нет GPU: Среда выполнения -> Сменить среду выполнения -> T4 GPU"
print(torch.cuda.get_device_name(0), round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1), "ГБ")

In [ ]:
# из /content: при повторном запуске в том же сеансе текущая папка – внутри удаляемого клона
%cd /content
!rm -rf /content/KinoCompass && git clone -q --depth 1 https://github.com/AlexTrav/KinoCompass.git /content/KinoCompass
!git -C /content/KinoCompass log --oneline -1
# те же версии, что в образе Docker (torch в Colab уже есть)
!pip install -q requests==2.34.2 pandas==3.1.0 pyarrow==26.0.1 scipy==1.18.1

In [ ]:
%cd /content/KinoCompass/model
!PYTHONPATH=/content/KinoCompass/data:/content/KinoCompass/research PERIOD={PERIOD} python -W ignore evaluate.py final_gpu

In [ ]:
from google.colab import files

!cd /content/KinoCompass/model && zip -q -r /content/final_gpu_{PERIOD}.zip reports/final_gpu && ls -la reports/final_gpu
files.download(f"/content/final_gpu_{PERIOD}.zip")